# Modeling Pipeline: Week Approach

## Runner Injury Risk Prediction — Lovdal et al. (2021) Replication

This notebook implements the **modeling pipeline for the week approach**. It mirrors
the day-approach pipeline (notebook 04) but operates on weekly aggregated features
with a binarized target (threshold 0.5, ADR-002).

### Key differences from day approach

| Aspect | Day (notebook 04) | Week (this notebook) |
|--------|-------------------|---------------------|
| Features | 7 days × 10 features = 70 | 3 weeks × 22 features + 3 ratios = 69 |
| Target | Binary (original) | Binary (binarized at 0.5) |
| Sentinel handling | −0.01 → 0.0 | Not needed |
| Paper benchmark | AUC-ROC **0.724** | AUC-ROC **0.678** |

### Pipeline

1. Load preprocessed week splits from `data/processed/`
2. Cross-validate all models with 5-fold GroupKFold (ADR-006)
3. Tune XGBoost hyperparameters (RandomizedSearchCV, 30 iterations)
4. Final evaluation on held-out test set
5. ROC/PR curves, confusion matrix, comparison table
6. SMOTE experiment (secondary, ADR-003)

**Paper benchmark**: AUC-ROC **0.678** (week approach, bagged XGBoost).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# Ensure src/ is importable regardless of the notebook working directory
def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ATHLETE_ID_COL, INJURY_COL, RANDOM_SEED
from src.data_loading import get_feature_columns
from src.modeling.evaluate import (
    compute_metrics,
    create_comparison_table,
    find_optimal_threshold,
    plot_confusion_matrix,
    plot_pr_curves,
    plot_roc_curves,
)
from src.modeling.models import create_model
from src.modeling.train import (
    cross_validate_model,
    train_final_model,
    tune_hyperparameters,
)
from src.preprocessing.common import get_feature_target_groups
from src.preprocessing.day_preprocessor import transform_scaled
from src.preprocessing.io import load_scaler, load_splits, save_model
from src.utils.logging_config import setup_logging
from src.utils.plotting import PALETTE, save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

---

## 1. Data Loading

We load the preprocessed week splits and scaler. The week target was already
binarized at threshold 0.5 in notebook 03 (ADR-002).

In [ ]:
# Load preprocessed splits and scaler
week_train, week_test = load_splits(prefix="week")
week_scaler = load_scaler(name="week_scaler")
feature_cols = get_feature_columns(week_train)

# Extract features, target, groups for cross-validation
X_train, y_train, groups_train = get_feature_target_groups(week_train, feature_cols)
X_test = week_test[feature_cols]
y_test = week_test[INJURY_COL]

# Scaled versions for Logistic Regression
X_train_scaled = transform_scaled(week_train, week_scaler, feature_cols)[feature_cols]
X_test_scaled = transform_scaled(week_test, week_scaler, feature_cols)[feature_cols]

# Class imbalance ratio (for XGBoost scale_pos_weight)
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
imbalance_ratio = n_neg / n_pos

print("Week approach — preprocessed data:")
print(f"  Train: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"  Test:  {X_test.shape[0]:,} rows × {X_test.shape[1]} features")
print(f"  Athletes: {groups_train.nunique()} train, {week_test[ATHLETE_ID_COL].nunique()} test")
print("\nClass balance (train) — binarized at threshold 0.5:")
print(f"  Negative: {n_neg:,} ({n_neg / len(y_train) * 100:.2f}%)")
print(f"  Positive: {n_pos:,} ({n_pos / len(y_train) * 100:.2f}%)")
print(f"  Imbalance ratio: {imbalance_ratio:.1f}:1")

---

## 2. Baseline: Dummy Classifier

Stratified Dummy classifier — sanity check, expected AUC-ROC ≈ 0.50.

In [ ]:
# --- Store all CV results for comparison ---
cv_results: dict[str, dict] = {}

# Dummy baseline
dummy_model = create_model("dummy")
dummy_cv = cross_validate_model(dummy_model, X_train, y_train, groups_train)

cv_results["Dummy"] = {
    metric: f"{dummy_cv[f'test_{metric}'].mean():.4f} ± {dummy_cv[f'test_{metric}'].std():.4f}"
    for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]
}
cv_results["Dummy"]["roc_auc_mean"] = dummy_cv["test_roc_auc"].mean()

print("Dummy Classifier — 5-fold GroupKFold CV:")
for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]:
    scores = dummy_cv[f"test_{metric}"]
    print(f"  {metric:>20s}: {scores.mean():.4f} ± {scores.std():.4f}")

---

## 3. Logistic Regression

Scaled features only — same rationale as day approach (notebook 04).

In [ ]:
# Logistic Regression on SCALED features
logreg_model = create_model("logistic_regression")
logreg_cv = cross_validate_model(logreg_model, X_train_scaled, y_train, groups_train)

cv_results["LogReg"] = {
    metric: f"{logreg_cv[f'test_{metric}'].mean():.4f} ± {logreg_cv[f'test_{metric}'].std():.4f}"
    for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]
}
cv_results["LogReg"]["roc_auc_mean"] = logreg_cv["test_roc_auc"].mean()

print("Logistic Regression — 5-fold GroupKFold CV (scaled features):")
for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]:
    scores = logreg_cv[f"test_{metric}"]
    print(f"  {metric:>20s}: {scores.mean():.4f} ± {scores.std():.4f}")

---

## 4. Random Forest

Unscaled features, `class_weight='balanced'`.

In [ ]:
# Random Forest on UNSCALED features
rf_label = "Random Forest"
rf_model = create_model("random_forest")
rf_cv = cross_validate_model(rf_model, X_train, y_train, groups_train)

cv_results[rf_label] = {
    metric: f"{rf_cv[f'test_{metric}'].mean():.4f} ± {rf_cv[f'test_{metric}'].std():.4f}"
    for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]
}
cv_results[rf_label]["roc_auc_mean"] = rf_cv["test_roc_auc"].mean()

print("Random Forest — 5-fold GroupKFold CV:")
for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]:
    scores = rf_cv[f"test_{metric}"]
    print(f"  {metric:>20s}: {scores.mean():.4f} ± {scores.std():.4f}")

---

## 5. XGBoost

Unscaled features, `scale_pos_weight` for class imbalance handling.

In [ ]:
# XGBoost on UNSCALED features
xgb_model = create_model("xgboost", imbalance_ratio=imbalance_ratio)
xgb_cv = cross_validate_model(xgb_model, X_train, y_train, groups_train)

cv_results["XGBoost"] = {
    metric: f"{xgb_cv[f'test_{metric}'].mean():.4f} ± {xgb_cv[f'test_{metric}'].std():.4f}"
    for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]
}
cv_results["XGBoost"]["roc_auc_mean"] = xgb_cv["test_roc_auc"].mean()

print(f"XGBoost — 5-fold GroupKFold CV (scale_pos_weight={imbalance_ratio:.1f}):")
for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]:
    scores = xgb_cv[f"test_{metric}"]
    print(f"  {metric:>20s}: {scores.mean():.4f} ± {scores.std():.4f}")

---

## 6. Cross-Validation Comparison

Side-by-side comparison with the paper benchmark (0.678 for the week approach).

In [ ]:
# Build comparison table
cv_display = {
    model: {k: v for k, v in metrics.items() if k != "roc_auc_mean"}
    for model, metrics in cv_results.items()
}
cv_table = pd.DataFrame(cv_display).T
cv_table.index.name = "Model"
print("Cross-Validation Results (mean ± std across 5 folds):\n")
print(cv_table.to_string())

In [ ]:
# Bar chart: AUC-ROC comparison
PAPER_BENCHMARK_WEEK = 0.678

model_names = list(cv_results.keys())
auc_means = [cv_results[m]["roc_auc_mean"] for m in model_names]

fig, ax = plt.subplots(figsize=(10, 5))
colors = [PALETTE["neutral"] if m == "Dummy" else PALETTE["primary"] for m in model_names]
bars = ax.bar(model_names, auc_means, color=colors, edgecolor="white", width=0.6)

# Paper benchmark line
ax.axhline(PAPER_BENCHMARK_WEEK, color=PALETTE["negative"], linestyle="--", linewidth=1.5,
           label=f"Paper benchmark ({PAPER_BENCHMARK_WEEK})")

for bar, val in zip(bars, auc_means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", fontweight="bold", fontsize=10)

ax.set_ylabel("AUC-ROC (CV mean)")
ax.set_title("Cross-Validation AUC-ROC — Week Approach", fontweight="bold")
ax.set_ylim(0, 1.0)
ax.legend(fontsize=10)
sns.despine()

save_figure(fig, "05_cv_auc_comparison", subdir="modeling", close=False)
plt.show()
plt.close(fig)

### Interpretation

- **Dummy ≈ 0.50**: confirms no information leakage.
- The week approach typically shows lower AUC-ROC than the day approach —
  consistent with the paper's finding (0.678 vs 0.724). Weekly aggregation
  smooths out the daily signal that may be predictive of acute injury onset.

---

## 7. Hyperparameter Tuning (XGBoost)

Same tuning strategy as notebook 04, with 30 iterations for local feasibility.

> **Production note**: `n_iter=30` for local M1 feasibility.
> In production environments, increase to **50+ iterations**.

In [ ]:
# Hyperparameter search space
param_distributions: dict = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5],
}

# Base model with imbalance ratio
xgb_base = create_model("xgboost", imbalance_ratio=imbalance_ratio)

# Tune — n_iter=30 for local feasibility (production: 50+)
xgb_tuned = tune_hyperparameters(
    xgb_base, X_train, y_train, groups_train,
    param_distributions=param_distributions,
    n_iter=30,
)

print("\nBest hyperparameters:")
for param, value in sorted(xgb_tuned.get_params().items()):
    if param in param_distributions:
        print(f"  {param}: {value}")

---

## 8. Final Evaluation on Test Set

All models retrained on full training set, evaluated on held-out test set.
The optimal classification threshold is selected on training set predictions
to avoid data leakage, then applied to the test set.

In [ ]:
# Train all models on full training set and collect test predictions
test_predictions: dict[str, tuple[np.ndarray, np.ndarray]] = {}
test_metrics: dict[str, dict[str, float]] = {}

xgb_label = "XGBoost (tuned)"

models_to_evaluate = {
    "Dummy": (create_model("dummy"), X_train, X_test),
    "LogReg": (create_model("logistic_regression"), X_train_scaled, X_test_scaled),
    rf_label: (create_model("random_forest"), X_train, X_test),
    "XGBoost (default)": (create_model("xgboost", imbalance_ratio=imbalance_ratio), X_train, X_test),
    xgb_label: (xgb_tuned, X_train, X_test),
}

for name, (model, X_tr, X_te) in models_to_evaluate.items():
    fitted = train_final_model(model, X_tr, y_train)
    y_prob = fitted.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    test_predictions[name] = (np.array(y_test), y_prob)
    test_metrics[name] = compute_metrics(np.array(y_test), y_pred, y_prob)

# Display test metrics
test_table = create_comparison_table(test_metrics)
print("Test Set Evaluation (threshold = 0.5):\n")
print(test_table[["auc_roc", "auc_pr", "recall", "precision", "f1", "brier_score"]].to_string())

In [ ]:
# Optimal threshold — selected on training set to avoid test leakage
y_true_test, y_prob_best = test_predictions[xgb_label]

# Compute train predictions for threshold selection (no test leakage)
y_prob_train_best = xgb_tuned.predict_proba(X_train)[:, 1]
optimal_threshold = find_optimal_threshold(np.array(y_train), y_prob_train_best, metric="f1")

y_pred_optimal = (y_prob_best >= optimal_threshold).astype(int)
metrics_optimal = compute_metrics(y_true_test, y_pred_optimal, y_prob_best)

print(f"\n{xgb_label} — Train-selected threshold: {optimal_threshold:.2f}")
print("\nMetrics at optimal threshold vs default (0.5):")
print(f"  {'Metric':>15s}  {'Optimal':>10s}  {'Default':>10s}")
print(f"  {'─' * 15}  {'─' * 10}  {'─' * 10}")
for metric in ["recall", "precision", "f1", "specificity"]:
    opt_val = metrics_optimal[metric]
    def_val = test_metrics[xgb_label][metric]
    print(f"  {metric:>15s}  {opt_val:>10.4f}  {def_val:>10.4f}")

---

## 9. ROC and PR Curves

In [ ]:
# ROC curves
fig_roc = plot_roc_curves(test_predictions, save_path=Path("modeling/05_roc_curves"))
fig_roc.axes[0].annotate(
    f"Paper benchmark: AUC = {PAPER_BENCHMARK_WEEK}",
    xy=(0.55, 0.05), fontsize=10, fontstyle="italic",
    color=PALETTE["negative"],
)
plt.show()
plt.close(fig_roc)

# PR curves
fig_pr = plot_pr_curves(test_predictions, save_path=Path("modeling/05_pr_curves"))
plt.show()
plt.close(fig_pr)

---

## 10. Confusion Matrix

In [ ]:
# Confusion matrix at optimal threshold
fig_cm = plot_confusion_matrix(
    y_true_test, y_pred_optimal,
    title=f"XGBoost (Tuned) — Week Approach (threshold={optimal_threshold:.2f})",
    save_path=Path("modeling/05_confusion_matrix"),
)
plt.show()
plt.close(fig_cm)

print("\nConfusion matrix breakdown:")
print(f"  True Negatives:  {metrics_optimal['tn']:,}")
print(f"  False Positives: {metrics_optimal['fp']:,}")
print(f"  False Negatives: {metrics_optimal['fn']:,}")
print(f"  True Positives:  {metrics_optimal['tp']:,}")

---

## 11. SMOTE Experiment (Secondary)

Same as notebook 04: imblearn Pipeline with SMOTE inside each CV fold (ADR-003).

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# SMOTE + XGBoost pipeline (scale_pos_weight=1.0 since SMOTE balances classes)
smote_xgb = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("model", create_model("xgboost", imbalance_ratio=1.0)),
])

smote_cv = cross_validate_model(smote_xgb, X_train, y_train, groups_train)

smote_auc = smote_cv["test_roc_auc"].mean()
weighting_auc = cv_results["XGBoost"]["roc_auc_mean"]

print("SMOTE + XGBoost — 5-fold GroupKFold CV:")
for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]:
    scores = smote_cv[f"test_{metric}"]
    print(f"  {metric:>20s}: {scores.mean():.4f} ± {scores.std():.4f}")

print("\nComparison:")
print(f"  Class weighting (XGBoost):  AUC-ROC = {weighting_auc:.4f}")
print(f"  SMOTE + XGBoost:            AUC-ROC = {smote_auc:.4f}")
print(f"  Difference:                 {smote_auc - weighting_auc:+.4f}")
print(f"\n→ ADR-003 {'confirmed' if weighting_auc >= smote_auc - 0.02 else 'needs review'}: "
      f"class weighting is {'competitive with' if abs(smote_auc - weighting_auc) < 0.02 else 'compared to'} SMOTE.")

---

## 12. Save Best Model

In [ ]:
# Retrain tuned XGBoost on full training set and save
best_model = train_final_model(xgb_tuned, X_train, y_train)
save_model(best_model, name="week_best_model")

print("Best model saved: week_best_model.pkl")
print(f"  Type: {type(best_model).__name__}")
print(f"  Test AUC-ROC: {test_metrics[xgb_label]['auc_roc']:.4f}")
print(f"  Optimal threshold: {optimal_threshold:.2f}")

---

## 13. Summary

### Pipeline flow

```
data/processed/week_train.parquet + week_test.parquet
  → load_splits("week")
  → Cross-validate: Dummy, LogReg (scaled), RF, XGBoost
  → Tune XGBoost (RandomizedSearchCV, 30 iterations)
  → Final evaluation on test set
  → Save best model → data/processed/week_best_model.pkl
```

### Key findings

1. **All real models beat the Dummy baseline** — the weekly aggregated features
   carry predictive signal for injury risk
2. **XGBoost** is the best performer, consistent with both the day approach and
   the paper's choice
3. **Week approach AUC-ROC is typically lower than the day approach** — this is
   consistent with the paper (0.678 vs 0.724). Weekly aggregation smooths out
   acute daily signals
4. **Class weighting vs SMOTE**: ADR-003 validated on both approaches — class
   weighting is simpler and competitive
5. **Paper benchmark comparison**: our result should be evaluated against the
   paper's 0.678 (bagged XGBoost)

### Cross-reference: Day vs Week

The day approach (notebook 04) generally achieves higher AUC-ROC because:
- **Daily granularity** captures acute load spikes (e.g., a single hard session
  followed by insufficient recovery)
- **7-day rolling window** × 10 features provides more temporal resolution than
  3-week × 22 features
- The week approach aggregates away within-week variance that may be predictive

A detailed comparison is in notebook 08.

### Next steps

→ **Notebook 06**: SHAP interpretability analysis
→ **Notebook 07**: Fairness analysis across proxy groups
→ **Notebook 08**: Day vs Week comparison and final recommendation